# Creational & Structural Patterns
Design patterns are time-tested solutions to recurring architectural problems in software engineering. In this first part, we explore Creational patterns (which deal with object creation mechanisms) and Structural patterns (which ease design by identifying simple ways to realize relationships between entities).

## 1. Singleton Pattern
The Singleton pattern ensures that a class has only one instance throughout the entire application lifecycle and provides a global point of access to it. It is commonly used for database connection pools, configuration managers, or loggers.

In [ ]:
class SingletonMeta(type):
    """
    A metaclass enabling Singleton behavior in Python classes.
    """
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            instance = super().__call__(*args, **kwargs)
            cls._instances[cls] = instance
        return cls._instances[cls]

class DatabaseConnection(metaclass=SingletonMeta):
    def __init__(self):
        self.connection = "Connected to DB"

# Usage
db1 = DatabaseConnection()
db2 = DatabaseConnection()
print(db1 is db2)  # True (Both variables point to the exact same memory instance)

## 2. Factory Pattern
The Factory pattern provides an interface for creating objects in a superclass, but allows subclasses or factory functions to alter the type of objects that will be created. It decouples object creation from business logic.

In [ ]:
from abc import ABC, abstractmethod

class Document(ABC):
    @abstractmethod
    def render(self) -> str:
        pass

class PDFDocument(Document):
    def render(self) -> str:
        return "Rendering PDF document."

class HTMLDocument(Document):
    def render(self) -> str:
        return "Rendering HTML document."

class DocumentFactory:
    @staticmethod
    def create_document(doc_type: str) -> Document:
        if doc_type == "pdf":
            return PDFDocument()
        elif doc_type == "html":
            return HTMLDocument()
        raise ValueError(f"Unknown document type: {doc_type}")

# Usage
doc = DocumentFactory.create_document("pdf")
print(doc.render())  # Rendering PDF document.

## 3. Builder Pattern
The Builder pattern separates the construction of a complex object from its representation, allowing you to produce different types and representations of an object using the exact same construction steps (often implemented with method chaining / fluent interfaces).

In [ ]:
class Computer:
    def __init__(self):
        self.cpu = None
        self.ram = None
        self.storage = None

    def __str__(self):
        return f"Computer [CPU: {self.cpu}, RAM: {self.ram}, Storage: {self.storage}]"

class ComputerBuilder:
    def __init__(self):
        self.computer = Computer()

    def set_cpu(self, cpu: str):
        self.computer.cpu = cpu
        return self

    def set_ram(self, ram: str):
        self.computer.ram = ram
        return self

    def set_storage(self, storage: str):
        self.computer.storage = storage
        return self

    def build(self) -> Computer:
        return self.computer

# Usage with method chaining
gaming_pc = (ComputerBuilder()
             .set_cpu("Intel i9")
             .set_ram("32GB")
             .set_storage("2TB NVMe")
             .build())
print(gaming_pc)

## 4. Adapter Pattern
The Adapter pattern allows objects with incompatible interfaces to collaborate. It acts as a wrapper that translates calls from one interface into a format expected by another system.

In [ ]:
# Old legacy system
class LegacyPrinter:
    def print_old_format(self, text: str) -> str:
        return f"Legacy Print: {text}"

# Modern expected interface
class ModernPrinter(ABC):
    @abstractmethod
    def print_data(self, data: str) -> str:
        pass

# Adapter wraps the legacy class
class PrinterAdapter(ModernPrinter):
    def __init__(self, legacy_printer: LegacyPrinter):
        self.legacy_printer = legacy_printer

    def print_data(self, data: str) -> str:
        return self.legacy_printer.print_old_format(data)

# Usage
adapter = PrinterAdapter(LegacyPrinter())
print(adapter.print_data("Hello World"))  # Legacy Print: Hello World

## 5. Proxy Pattern
The Proxy pattern provides a surrogate or placeholder object for another object to control access to it (e.g., for lazy loading, access control, or logging).

In [ ]:
class Database(ABC):
    @abstractmethod
    def query(self, sql: str) -> str:
        pass

class RealDatabase(Database):
    def query(self, sql: str) -> str:
        return f"Executing query: {sql}"

class SecurityProxy(Database):
    def __init__(self, real_db: RealDatabase, user_role: str):
        self.real_db = real_db
        self.user_role = user_role

    def query(self, sql: str) -> str:
        if self.user_role != "admin" and "DROP" in sql.upper():
            raise PermissionError("Unauthorized: Non-admins cannot execute DROP queries!")
        return self.real_db.query(sql)

# Usage
proxy = SecurityProxy(RealDatabase(), user_role="guest")
try:
    proxy.query("DROP TABLE users;")
except PermissionError as e:
    print(e)  # Unauthorized: Non-admins cannot execute DROP queries!

## 6. Facade Pattern
The Facade pattern provides a simplified, unified interface to a complex subsystem or framework, hiding its internal complexity from the client code.

In [ ]:
class CPU:
    def freeze(self): print("CPU freezing...")
    def jump(self, position): print(f"CPU jumping to {position}...")

class Memory:
    def load(self, position, data): print(f"Loading data into memory at {position}...")

class HardDrive:
    def read(self, lba, size): return f"Data from sector {lba}"

class ComputerFacade:
    """Facade hiding the messy subsystem initialization steps."""
    def __init__(self):
        self.cpu = CPU()
        self.memory = Memory()
        self.hard_drive = HardDrive()

    def start_computer(self):
        self.cpu.freeze()
        self.memory.load(0, self.hard_drive.read(100, 1024))
        self.cpu.jump(0)
        print("Computer started successfully.")

# Usage
computer = ComputerFacade()
computer.start_computer()

## 7. Decorator Pattern
The Decorator pattern dynamically attaches new behaviors to objects by placing them inside wrapper objects containing those behaviors. (Note: Python also features built-in function decorators using the @ syntax, which apply a similar concept at the function level).

In [ ]:
class Coffee(ABC):
    @abstractmethod
    def cost(self) -> float:
        pass

class SimpleCoffee(Coffee):
    def cost(self) -> float:
        return 2.00

class CoffeeDecorator(Coffee):
    def __init__(self, coffee: Coffee):
        self._coffee = coffee

    def cost(self) -> float:
        return self._coffee.cost()

class Milk(CoffeeDecorator):
    def cost(self) -> float:
        return super().cost() + 0.50

class Sugar(CoffeeDecorator):
    def cost(self) -> float:
        return super().cost() + 0.20

# Usage: Decorating simple coffee with milk and sugar
my_coffee = Sugar(Milk(SimpleCoffee()))
print(f"Total cost: ${my_coffee.cost():.2f}")  # Total cost: $2.70

### Best Practices & Common Pitfalls
**Avoid Over-Engineering:** Design patterns are tools, not rules. Do not force patterns (like Singletons or Factories) into simple scripts where basic classes or functions suffice.

**Understand Pythonic Alternatives:** Many structural patterns in C++ or Java (like lightweight iteration wrappers or function overloading adapters) can be achieved more cleanly in Python using first-class functions, closures, or built-in iterators.